# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [2]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

Fichero: c:\Users\claud\master\repos\icai-muiaap-operacion-de-modelos1\semana3\assets\03-wine-quality\inference_samples.csv
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Filas de ejemplo: 5


## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Tener que volver a entrenar el modelo para utilizarlo |
| Versión del esquema | manifest.json | Utilizar un contrato con una estructura incompatible|
| Orden y nombres de features | manifest.json | introducir variables incorrectas o en distinto orden |
| Versión del preprocesado | manifest.json | aplicar un preprocesado incompatible con el modelo |
| Etiquetas de salida permitidas | manifest.json | devolver categorias no reconocidads por el sistema |
| Versión del modelo | manifest.json| utilizar una version del modelo distinta de la esperada |
| Tipo de estimador | manifest.json | cargar un estimador incompatible con la interfaz requerida. |

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [8]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-demo-v1"

manifest_draft = {
    "schema_version": "1.0.0", #`porque es la primera version`
    "model_version": legacy_model_version,
    "preprocessing_version": "1.0.0",
    "feature_names": feature_names, #hay 11 valores sin contar sample_id
    "output_labels": ["low", "medium", "high"], #representa las categorias que la interfaz podrá devolver
    "estimator_type": "RandomForestClassifier",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))

print(
    "\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas."
    "\n\n1. Si falta manifest.json, no se debería cargar el modelo, porque no tenemos la información necesaria para comprobar que es compatible."
    "\n\n2. Si aparecen features desconocidas o en otro orden, se debería mostrar un error y no hacer la predicción, ya que el modelo espera unas variables concretas en un orden determinado."
    "\n\n3. Si el modelo devuelve una etiqueta que no está en output_labels, se debería rechazar esa predicción porque el resultado no coincide con las categorías que hemos definido en el manifiesto."
    "\n\n4. Si una fila del CSV no cumple el contrato, se debería rechazar el archivo completo e indicar qué fila tiene el error, para evitar guardar resultados incompletos."
)

{
  "schema_version": "1.0.0",
  "model_version": "wine-quality-rf-demo-v1",
  "preprocessing_version": "1.0.0",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "low",
    "medium",
    "high"
  ],
  "estimator_type": "RandomForestClassifier"
}
{
  "schema_version": "1.0.0",
  "model_version": "wine-quality-rf-demo-v1",
  "preprocessing_version": "1.0.0",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "low",
    "medium",
    "high"
  ],
  "estimator_type": "RandomForestClassifier"
}

La estructura es válida. Sustituye los TODO y las etiquetas provisionales p

## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` |Validacion de entrada, antes de predecir | el orden de las columnas no coincide con el esperado | No |
| Cambia `preprocessing_version` | validacion del manifiesto, antes de cargar el modelo |la version del preprocesado no es compatible | No |
| El modelo devuelve `unknown` |validacion de salida, despues de predecir | la etiqueta unknown no esta entre las permitidas | No |
| La petición incluye `ph = 99` | Validacion de entrada, antes de predecir | el valor de ph esta fuera del rango permitido | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
Los seis campos del manifiesto (shcema_version, model_version, preprocessing_version, feature_names, output_labels y estimator_type) deberian ser obligatorios. No pondria valores por defecto porque, si falta alguno, podríamos asumir información incorrecta sobre el modelo.

- ¿Aceptarás campos desconocidos en el manifiesto?
No, rechazaría los campos desconocidos para asegurar que el manifiesto sigue la estructura que hemos definido y evitar errores. 

- ¿Validarás el manifiesto antes de cargar el objeto serializado?
Si, primero comprobaria que el manifiesto exite y que sus cmapos son correctos y compatibles. Si hay algun error, no cargaría el modelo.

- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
Lo escribiría solo despues de validar todas las filas y comprobar que las predicciones son correctas, para evitar generar archivos incompletos. 

- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
Mostraría un mensaje de error indicando que ha falladoy por qué. Por ejemplo: "Error: el orden de las columnas no coincide con el esperado por el modelo" 

- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?
Se rechazaría el CSV completo y no se guardaría ninguna predicción, aunque la primera fila fuera correcta. Así evitamos publicar resultados parciales. 

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.

In [10]:
#esto es para ver lo que hay dentro del modelo de la semana 3
import joblib

model_path = week_root / "assets/04-model-packaging/wine_quality_classifier.joblib"

payload = joblib.load(model_path)

print("Versión del modelo:", payload["model_version"])
print("Features:", payload["feature_names"])

estimator = payload["estimator"]

print("Tipo de estimador:", type(estimator).__name__)
print("Etiquetas:", estimator.classes_)

Versión del modelo: wine-quality-rf-v1
Features: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Tipo de estimador: RandomForestClassifier
Etiquetas: ['acceptable' 'excellent' 'needs_review']
